# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# Setup for the fixed FlyRank warehouse release.
# The token is read at runtime; never paste it into the notebook source.

import os
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)
APRIL = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse paths ready.")
print("Feature window: March 1-31, 2026")
print("Outcome window: April 1-30, 2026")


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# EXACTLY THREE formal verification queries for the Assignment 4 contract.

# Query 1 — raw grain: zero rows means no duplicate
# (report_date, client_hash_id, content_hash_id) keys.
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1 — grain check")
display(grain_check)


# Query 2 — size and date span of the March feature partition.
march_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS pages,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

print("\nQUERY 2 — March size and date span")
display(march_check)


# Query 3 — availability. Use IS TRUE because availability flags are
# not safely treated as ordinary two-valued booleans.
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {MARCH}
""").df()

print("\nQUERY 3 — March availability")
display(availability_check)


### Design analysis 1 — choose the minimum usable-day requirement

This analysis is **not a fourth formal verification query**. It is a design analysis used to choose the page-coverage rule from the data rather than assume a percentage in advance.

A usable day means that the page has a daily row with `gsc_data_available IS TRUE`. A zero-impression day can still be a valid observed day, so the rule does **not** require `gsc_impressions > 0`.

The fixed March release contains **176,738 pages with at least one genuinely observed GSC day**. Those pages account for **3,611,061 GSC-observed daily rows**, an average of about **20.43 usable days per observed page**.

An independently executed query on this same release found **106,546 pages with at least 20 usable GSC days in March**. This retains about **60.28% of the genuinely GSC-observed March page population** while requiring approximately two-thirds of the month.

A separate executed April coverage check also found a **20-day median** in its downstream page cohort. Taken together, these results support a symmetric proof-of-concept rule of **at least 20 usable GSC days in March for feature eligibility and at least 20 usable GSC days in April for outcome observability**.

The value **20 was therefore determined from coverage analysis**, not chosen as a 90% completeness requirement or assumed beforehand. It is a practical proof-of-concept threshold: strong enough to avoid representing a month using only a handful of days, while still retaining a large page population. The effect of this sampling rule remains a stated limitation.


In [ ]:
# Reproduce the March coverage evidence used to choose the cutoff.

march_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

march_coverage_summary = pd.DataFrame({
    "measure": [
        "pages_with_any_gsc_day",
        "mean_usable_days",
        "pages_with_at_least_20_days",
        "pct_observed_pages_retained_at_20"
    ],
    "value": [
        len(march_coverage),
        march_coverage["march_usable_days"].mean(),
        int((march_coverage["march_usable_days"] >= 20).sum()),
        100.0 * (march_coverage["march_usable_days"] >= 20).mean()
    ]
})

display(march_coverage_summary)


In [ ]:
# Compare candidate March coverage cutoffs.
# This shows the trade-off instead of presenting 20 days as a magic value.

candidate_cutoffs = [7, 14, 18, 20, 21, 24, 27, 28, 30, 31]

march_cutoff_analysis = pd.DataFrame([
    {
        "minimum_days": cutoff,
        "pages_retained": int(
            (march_coverage["march_usable_days"] >= cutoff).sum()
        ),
        "pct_of_gsc_observed_pages_retained": round(
            100.0 * (
                march_coverage["march_usable_days"] >= cutoff
            ).mean(),
            2
        ),
        "pct_of_march_window_required": round(
            100.0 * cutoff / 31,
            2
        )
    }
    for cutoff in candidate_cutoffs
])

display(march_cutoff_analysis)


In [ ]:
# Apply the same 20-day observability idea to April and inspect the
# March→April overlap. April remains outcome-side information only:
# it is NOT used as a March predictive feature.

april_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

coverage_pair = march_coverage.merge(
    april_coverage,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

coverage_pair["april_usable_days"] = (
    coverage_pair["april_usable_days"]
    .fillna(0)
    .astype(int)
)

coverage_decision = pd.DataFrame([{
    "minimum_usable_days": 20,
    "march_feature_eligible_pages": int(
        (coverage_pair["march_usable_days"] >= 20).sum()
    ),
    "march_and_april_observable_pages": int(
        (
            (coverage_pair["march_usable_days"] >= 20)
            &
            (coverage_pair["april_usable_days"] >= 20)
        ).sum()
    ),
    "clients_in_joint_observable_population": int(
        coverage_pair.loc[
            (coverage_pair["march_usable_days"] >= 20)
            &
            (coverage_pair["april_usable_days"] >= 20),
            "client_hash_id"
        ].nunique()
    )
}])

display(coverage_decision)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.